# Untold Lores — Image Generation Server
Runs FLUX.1-schnell (text→image) as an API.

**Steps:**
1. Enable GPU: Settings → Accelerator → **GPU T4 x2**
2. Add your ngrok token to Kaggle Secrets as `NGROK_TOKEN` (or paste it directly in Cell 4)
3. Run all cells in order
4. Copy `LOCAL_MODEL_URL=...` into your `.env` file
5. Run `npm run images` on your laptop

In [ ]:
# Cell 1 — Install dependencies
!pip install -q diffusers transformers accelerate flask pyngrok sentencepiece protobuf

In [ ]:
# Cell 2 — Load FLUX.1-schnell
import torch
from diffusers import FluxPipeline
import io, tempfile

print('Loading FLUX.1-schnell...')
flux_pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.bfloat16
)
flux_pipe.enable_model_cpu_offload()
print('FLUX ready!')

In [ ]:
# Cell 3 — Start API server
from flask import Flask, request, send_file, jsonify
import threading

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

@app.route('/txt2img', methods=['POST'])
def txt2img():
    data = request.json
    prompt = data.get('prompt', 'cinematic scene')
    seed   = int(data.get('seed', 42))
    width  = int(data.get('width', 1280))
    height = int(data.get('height', 720))

    generator = torch.Generator().manual_seed(seed)
    result = flux_pipe(
        prompt=prompt,
        width=width,
        height=height,
        guidance_scale=0.0,
        num_inference_steps=4,
        generator=generator,
    )
    image = result.images[0]

    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=90)
    buf.seek(0)
    return send_file(buf, mimetype='image/jpeg')

t = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, threaded=False))
t.daemon = True
t.start()
print('Flask server started on port 5000')

In [ ]:
# Cell 4 — Expose with ngrok
from pyngrok import ngrok, conf

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret('NGROK_TOKEN')
except Exception:
    ngrok_token = 'PASTE_YOUR_NGROK_TOKEN_HERE'

conf.get_default().auth_token = ngrok_token

# Kill any existing tunnels before opening a new one
ngrok.kill()

public_url = ngrok.connect(5000).public_url
print('=' * 60)
print('Server is live!')
print(f'LOCAL_MODEL_URL={public_url}')
print('=' * 60)

In [ ]:
# Cell 5 — Keep session alive
import time
print('Keeping session alive... (interrupt kernel when done)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)